# AcuDockMetal
**Metal-Aware ε-Certified Molecular Docking**

1. Run **Cell 1** (installs packages, then runtime restarts automatically)
2. Run **Cell 2** (launches the interactive UI)
3. Use the tabbed panel to dock, scan chelators, visualize, or browse the metal library

In [ ]:
#@title Step 1: Install Dependencies (runtime restarts after this cell)
!pip install -q vina meeko gemmi rdkit prody py3Dmol openbabel-wheel \
    pdbfixer openmm biopython pandas numpy scipy ipywidgets matplotlib seaborn
# GNINA v1.3.2 for CNN rescoring
!wget -q https://github.com/gnina/gnina/releases/download/v1.3.2/gnina.1.3.2 \
    -O /usr/local/bin/gnina && chmod +x /usr/local/bin/gnina \
    && echo 'GNINA v1.3.2 installed' \
    || echo 'GNINA not available -- will use Vina only'
# Clone / update repo
!git clone https://github.com/Grimlock5310/AcuDockMetal.git /content/AcuDockMetal 2>/dev/null \
    || (cd /content/AcuDockMetal && git pull)
# Restart runtime so newly installed native modules are loadable
import os; os.kill(os.getpid(), 9)

In [ ]:
#@title Step 2: Launch Interactive UI
%cd /content/AcuDockMetal

import os, sys, logging, urllib.request, warnings
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

import ipywidgets as w
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt

from acudockmetal import (
    MetalParameterLibrary, ChelatingGroupDetector, MetalSiteDetector,
    CoordinationHypothesisGenerator, AcuDockMetalPipeline,
    DockingVisualizer, GeometryValidator,
)
from acudockmetal.pipeline import AcuDockConfig

metal_lib = MetalParameterLibrary()
chelator = ChelatingGroupDetector(min_confidence=0.3)

# ---- State ----
_state = {'result': None, 'pipeline': None}

# ===================================================================
# Header
# ===================================================================
header = w.HTML(
    '<h2 style="margin:0">AcuDockMetal</h2>'
    '<p style="color:#666;margin:0 0 8px">Metal-Aware ε-Certified Molecular Docking &mdash; '
    f'Supports: <b>{", ".join(metal_lib.supported_metals)}</b></p>'
)

# ===================================================================
# Tab 1 — Dock
# ===================================================================
pdb_input   = w.Text(value='1T6E', description='PDB ID:', placeholder='e.g. 1T6E',
                     style={'description_width': '100px'}, layout=w.Layout(width='360px'))
smi_input   = w.Textarea(value='ONC(=O)c1cc(Cc2ccccc2)on1', description='Ligand SMILES:',
                         rows=2, style={'description_width': '100px'}, layout=w.Layout(width='460px'))
exhaust_sl  = w.IntSlider(value=32, min=4, max=128, step=4, description='Exhaustiveness:',
                          style={'description_width': '100px'}, layout=w.Layout(width='360px'))
nposes_sl   = w.IntSlider(value=20, min=5, max=100, step=5, description='Num poses:',
                          style={'description_width': '100px'}, layout=w.Layout(width='360px'))
eps_sl      = w.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Epsilon (ε):',
                            style={'description_width': '100px'}, layout=w.Layout(width='360px'))
ph_sl       = w.FloatSlider(value=7.4, min=4.0, max=10.0, step=0.1, description='pH:',
                            style={'description_width': '100px'}, layout=w.Layout(width='360px'))
certify_cb  = w.Checkbox(value=True, description='ε-Certification', indent=False)
gnina_cb    = w.Checkbox(value=True, description='Use GNINA', indent=False)
taut_cb     = w.Checkbox(value=True, description='Enumerate tautomers', indent=False)
rescore_dd  = w.Dropdown(options=[('L1: Metal-aware', 1), ('L2: + CNN', 2),
                                   ('L3: + MM minimization', 3), ('L4: + QM/MM', 4)],
                         value=3, description='Rescore level:',
                         style={'description_width': '100px'}, layout=w.Layout(width='360px'))

dock_btn    = w.Button(description='Run Docking', button_style='success',
                       icon='play', layout=w.Layout(width='200px', height='40px'))
dock_out    = w.Output(layout=w.Layout(border='1px solid #ddd', padding='8px',
                                       max_height='500px', overflow_y='auto'))

def _run_docking(btn):
    dock_out.clear_output()
    with dock_out:
        pdb_id = pdb_input.value.strip().upper()
        pdb_path = f'{pdb_id}.pdb'
        if not os.path.exists(pdb_path):
            print(f'Downloading {pdb_id}...')
            urllib.request.urlretrieve(
                f'https://files.rcsb.org/download/{pdb_id}.pdb', pdb_path)

        config = AcuDockConfig(
            exhaustiveness=exhaust_sl.value,
            num_poses=nposes_sl.value,
            epsilon=eps_sl.value,
            ph=ph_sl.value,
            run_certification=certify_cb.value,
            gnina_enabled=gnina_cb.value,
            enumerate_tautomers=taut_cb.value,
            max_rescore_level=rescore_dd.value,
        )
        pipeline = AcuDockMetalPipeline(config)
        _state['pipeline'] = pipeline

        print(f'Docking {smi_input.value.strip()} into {pdb_id}...')
        result = pipeline.run(
            receptor_path=pdb_path,
            ligand_input=smi_input.value.strip(),
            output_dir=f'./output_{pdb_id}',
        )
        _state['result'] = result
        print(result.summary)

dock_btn.on_click(_run_docking)

dock_tab = w.VBox([
    w.HBox([w.VBox([pdb_input, smi_input]),
            w.VBox([exhaust_sl, nposes_sl, eps_sl, ph_sl, rescore_dd])]),
    w.HBox([certify_cb, gnina_cb, taut_cb]),
    dock_btn, dock_out,
])

# ===================================================================
# Tab 2 — Chelator Scanner
# ===================================================================
chel_smi    = w.Textarea(value='ONC(=O)c1ccccc1', description='SMILES:',
                         rows=2, style={'description_width': '60px'}, layout=w.Layout(width='460px'))
chel_metal  = w.Dropdown(options=metal_lib.supported_metals, value='ZN',
                         description='Metal:', style={'description_width': '60px'})
chel_btn    = w.Button(description='Scan', button_style='info', icon='search',
                       layout=w.Layout(width='120px'))
chel_out    = w.Output(layout=w.Layout(border='1px solid #ddd', padding='8px'))

def _scan_chelators(btn):
    chel_out.clear_output()
    with chel_out:
        try:
            donors = chelator.detect_from_smiles(chel_smi.value.strip())
            print(chelator.summary(donors))
            compat = chelator.compatible_with_metal(donors, chel_metal.value)
            print(f'\n{chel_metal.value}-compatible: {len(compat)} group(s), '
                  f'denticity = {chelator.total_denticity(compat)}')
        except Exception as e:
            print(f'Error: {e}')

chel_btn.on_click(_scan_chelators)
chel_tab = w.VBox([w.HBox([chel_smi, chel_metal, chel_btn]), chel_out])

# ===================================================================
# Tab 3 — Visualize
# ===================================================================
viz_dd  = w.Dropdown(options=['Metal Site (3D)', 'Best Pose (3D)',
                              'Score Distribution', 'Energy Decomposition',
                              'Coordination Radar', 'Certification Convergence',
                              'Rescoring Funnel'],
                     value='Metal Site (3D)', description='View:',
                     style={'description_width': '50px'}, layout=w.Layout(width='300px'))
viz_btn = w.Button(description='Show', button_style='primary', icon='eye',
                   layout=w.Layout(width='120px'))
viz_out = w.Output(layout=w.Layout(border='1px solid #ddd', padding='8px',
                                   min_height='200px'))

def _show_viz(btn):
    viz_out.clear_output()
    with viz_out:
        r = _state.get('result')
        p = _state.get('pipeline')
        if r is None or p is None:
            print('Run a docking job first (Dock tab).')
            return
        vis = p.visualizer
        choice = viz_dd.value
        try:
            if choice == 'Metal Site (3D)' and r.metal_sites:
                v = vis.view_metal_site(r.prepared_receptor.pdb_path, r.metal_sites[0])
                v.show()
            elif choice == 'Best Pose (3D)' and r.docking_poses:
                hyp = r.hypotheses[0] if r.hypotheses else None
                v = vis.view_docked_pose(r.prepared_receptor.pdb_path,
                                        r.docking_poses[0], hypothesis=hyp)
                v.show()
            elif choice == 'Score Distribution' and r.docking_poses:
                fig = vis.plot_score_distribution(r.docking_poses)
                plt.show()
            elif choice == 'Energy Decomposition' and r.metal_scores:
                fig = vis.plot_energy_decomposition(r.metal_scores)
                plt.show()
            elif choice == 'Coordination Radar' and r.metal_scores:
                fig = vis.plot_metal_geometry_radar(r.metal_scores[0])
                plt.show()
            elif choice == 'Certification Convergence' and r.certification:
                fig = vis.plot_certification_convergence(r.certification)
                plt.show()
            elif choice == 'Rescoring Funnel' and r.rescoring:
                fig = vis.plot_multifidelity_funnel(r.rescoring)
                plt.show()
            else:
                print(f'No data available for "{choice}". Run docking first or check results.')
        except Exception as e:
            print(f'Visualization error: {e}')

viz_btn.on_click(_show_viz)
viz_tab = w.VBox([w.HBox([viz_dd, viz_btn]), viz_out])

# ===================================================================
# Tab 4 — Metal Library
# ===================================================================
lib_dd  = w.Dropdown(options=metal_lib.supported_metals, value='ZN',
                     description='Metal:', style={'description_width': '50px'})
lib_out = w.Output(layout=w.Layout(border='1px solid #ddd', padding='8px'))

def _show_metal(change):
    lib_out.clear_output()
    with lib_out:
        mp = metal_lib[change['new']]
        print(f'{mp.name} ({mp.symbol})')
        print(f'  Oxidation states: {mp.common_oxidation_states}')
        print(f'  CN range: {mp.typical_cn_range[0]}-{mp.typical_cn_range[1]}')
        print(f'  Preferred donors: {mp.preferred_donors}')
        print(f'  Water propensity: {mp.water_propensity}')
        print(f'  Ideal distances:')
        for donor, d in mp.ideal_distances.items():
            print(f'    {donor}: {d:.2f} A')
        print(f'  Geometries:')
        for cn, geoms in mp.preferred_geometries.items():
            print(f'    CN={cn}: {", ".join(geoms)}')
        print(f'  12-6-4 LJ: r_min={mp.lj_r_min}, eps={mp.lj_epsilon}, C4={mp.c4_coefficient}')
        if mp.notes:
            print(f'  Notes: {mp.notes}')

lib_dd.observe(_show_metal, names='value')
# Trigger initial display
lib_out.clear_output()
with lib_out:
    mp = metal_lib['ZN']
    print(f'{mp.name} ({mp.symbol})')
    print(f'  Oxidation states: {mp.common_oxidation_states}')
    print(f'  CN range: {mp.typical_cn_range[0]}-{mp.typical_cn_range[1]}')
    print(f'  Preferred donors: {mp.preferred_donors}')
    for cn, geoms in mp.preferred_geometries.items():
        print(f'    CN={cn}: {", ".join(geoms)}')
    if mp.notes:
        print(f'  Notes: {mp.notes}')

lib_tab = w.VBox([lib_dd, lib_out])

# ===================================================================
# Assemble Tabs
# ===================================================================
tabs = w.Tab(children=[dock_tab, chel_tab, viz_tab, lib_tab])
tabs.set_title(0, 'Dock')
tabs.set_title(1, 'Chelator Scanner')
tabs.set_title(2, 'Visualize')
tabs.set_title(3, 'Metal Library')

display(header, tabs)